# 05. Feature selection y reducción de dimensionalidad

**Fases del guía metodológica cubiertas: 10 (Feature selection y reducción dimensional)**

> Regla central aplicada: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.
> El test nunca influye en preprocessing, selección de variables, hiperparámetros o elección de modelo.



## 10.1 Selección por reglas

- Eliminadas `G1, G2, G3` (post-evento / no disponibles en producción) — hecho en fase 5.
- Eliminado `Walc` de las features (es el target).
- Sin IDs, sin variables prohibidas.

### 10.1.1 Carga de los datos de entrenamiento

Cargamos los conjuntos procesados y aplicamos el feature engineering de la fase 9 para
trabajar con las 37 columnas candidatas. Todo el análisis de selección de esta fase se
realiza **solo sobre train**: cualquier decisión basada en validation/test contaminaría
la evaluación final.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_data import load_processed
X_train, X_val = load_processed()["X_train"], load_processed()["X_val"]
y_train, y_val = load_processed()["y_train"], load_processed()["y_val"]
from src.features.build_features import add_domain_features
Xtr = add_domain_features(X_train)
Xva = add_domain_features(X_val)
print("Features:", Xtr.shape[1])


Features: 37



## 10.2 Métodos filtro (solo sobre train)

### 10.2.1 Varianza baja

Las columnas con varianza prácticamente nula no aportan información (son casi constantes).
Aplicamos `VarianceThreshold` sobre las variables numéricas/ordinales del train para
detectarlas. En este dataset esperamos no encontrar ninguna, porque el cuestionario ya
fue depurado, pero es una comprobación barata que evita incluir ruido puro en el modelo.


In [2]:

from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
import pandas as pd

# Varianza baja (sobre features numéricas/ordinales estandarizadas aproximadas)
sel = VarianceThreshold(threshold=0.01)
mask = sel.fit(Xtr.select_dtypes(include=["int64", "float64"]))
print("Features con varianza ~0:", [c for c, v in zip(
    Xtr.select_dtypes(include=["int64", "float64"]).columns,
    sel.variances_) if v < 0.01])


Features con varianza ~0: []



### 10.2.2 Información mutua con el target

La **información mutua** captura relaciones lineales y no lineales entre cada feature y
el target (a diferencia de Pearson). La calculamos sobre train, codificando primero las
categóricas con el preprocessor ajustado solo con train (porque la función exige datos
numéricos). El ranking resultante nos dice qué variables, por sí solas, contienen más
información sobre el consumo alto: esperamos que `goout` y `Dalc` dominen el ranking.


In [3]:

# Información mutua con el target (SÓLO train): codificamos con el preprocessor
# ajustado solo con train (las features categóricas requieren datos numéricos)
from src.models.train_model import get_preprocessor
pre_mi = get_preprocessor()
Xtr_enc = pre_mi.fit_transform(Xtr)
names_mi = pre_mi.get_feature_names_out()
mi = mutual_info_classif(Xtr_enc, y_train, random_state=42)
mi_df = pd.DataFrame({"feature": names_mi, "mi": mi}).sort_values("mi", ascending=False)
mi_df.head(12)


,feature,mi
53,ord__Dalc,0.178003
23,cat__Fjob_teacher,0.071940
52,ord__goout,0.068655
51,ord__freetime,0.051355
9,cat__address_U,0.048327
6,cat__sex_F,0.042647
0,num__age,0.039825
8,cat__address_R,0.038238
10,cat__famsize_GT3,0.033719
19,cat__Fjob_at_home,0.032105



## 10.4 Métodos embedded (importancia en train)

### 10.4.1 Importancia por Gini de un LightGBM

Los modelos de árboles proporcionan una **importancia intrínseca** (ganancia media de
impureza). Entrenamos un LightGBM rápido sobre train (dentro de un pipeline, para
respetar la codificación) y mostramos las 15 features con mayor importancia. Este ranking
es orientativo: la importancia por Gini puede favorecer a variables numéricas con muchos
valores, por lo que lo contrastaremos con la **permutation importance** sobre validation.


In [4]:

from src.models.train_model import make_pipeline, _lightgbm
from sklearn.metrics import roc_auc_score
import numpy as np

pipe = make_pipeline(_lightgbm(42))
pipe.fit(Xtr, y_train)
imp = pipe.named_steps["model"].feature_importances_
names_imp = pipe.named_steps["preprocessor"].get_feature_names_out()
imp_df = pd.DataFrame({"feature": names_imp, "importance": imp}).sort_values("importance", ascending=False)
imp_df.head(15)


,feature,importance
3,num__absences,272
52,ord__goout,174
53,ord__Dalc,170
54,ord__health,154
1,num__Medu,122
0,num__age,120
2,num__Fedu,112
48,ord__studytime,99
50,ord__famrel,85
47,ord__traveltime,82



### 10.4.2 Permutation importance sobre validation

La **permutation importance** mide cuánto empeora la métrica (ROC-AUC) al permutar
aleatoriamente cada feature: si al romper una columna el modelo pierde rendimiento, esa
columna es informativa; si no cambia nada, es prescindible. La calculamos sobre
**validation** (el test sigue bloqueado) con el pipeline ya entrenado. Este es el método
más fiable de los tres y el que usaremos para razonar la decisión final.


In [5]:

# Permutation importance sobre VALIDATION (el test queda bloqueado)
from sklearn.inspection import permutation_importance
r = permutation_importance(pipe, Xva, y_val, n_repeats=10, random_state=42, scoring="roc_auc")
perm = pd.DataFrame({"feature": Xtr.columns, "imp_mean": r.importances_mean,
                     "imp_std": r.importances_std}).sort_values("imp_mean", ascending=False)
perm.head(12)


,feature,imp_mean,imp_std
26,Dalc,0.165266,0.025790
25,goout,0.055793,0.024137
6,Medu,0.007051,0.005295
1,sex,0.006529,0.004650
13,studytime,0.006054,0.007548
28,absences,0.005935,0.011170
10,reason,0.005176,0.009195
11,guardian,0.004820,0.007025
23,famrel,0.002683,0.003775
18,activities,0.002469,0.003684



## 10.5 Decisión

- La dimensionalidad es baja (37 features engineered) y las correlaciones altas son
  tolerables para árboles. **No se descarta ninguna feature por criterio estadístico**;
  el CV con pipeline completo (fase 13) validará el conjunto.
- `PCA/UMAP` no aplican: la interpretabilidad prima y el dataset es pequeño.
- La selección final se confirma con el CV de la fase 13 (dentro de folds).

### 10.5.1 Guardado de la lista de features

Persistimos el orden de columnas en `configs/final_features.json`: este fichero sirve de
**contrato entre entrenamiento y producción** (la API lo usa para reconstruir el
DataFrame de entrada). Cualquier cambio futuro en el orden de features debe pasar por
este fichero y por el reentrenamiento del pipeline.


In [6]:

import json
(ROOT / "configs").mkdir(exist_ok=True)
feature_list = Xtr.columns.tolist()
(ROOT / "configs" / "final_features.json").write_text(json.dumps(feature_list, indent=2), encoding="utf-8")
print("Lista de features final guardada:", len(feature_list))


Lista de features final guardada: 37
